# PKG Attrition — Source EDA v5 · deposit panel & account-level payment join

**What changed since v4.** The deposit source is different. v2–v4 ran against an
aggregated, already-flagged customer extract. This runs against the **daily
account-level deposit table**, which carries `acct_status`, `acct_status_desc`,
`opened_dt`, `closed_dt`, `deposit_family` and `avg_monthly_bal_1`. That changes
three things:

| Brief's assumption | Reality now | Consequence |
|---|---|---|
| §12 "no closure or dormancy flag is available" | `acct_status` exists; `'C'` = closed | The decline proxy can be **scored against a real label** instead of standing in for one |
| Deposits are a customer-month series | Account-day rows, one per business day | Grain, panel completeness and post-closure behaviour all have to be established first |
| `balance` semantics unknown | End-of-day balance | Month-end vs monthly-average is now our choice, not a question |

**What is carried over as known, not re-derived** (from the v3/v4 runs):
rail taxonomy, category→direction 1:1, counterparty key provenance, 5C/6C
duplication. `RUN_TXN_TAXONOMY = False` keeps those cells off. Turn it on only
if the source schema changed.

### Conventions in this notebook

- **No `.show()`.** Every cell ends on a single pandas DataFrame so one
  screenshot captures the whole result. Multiple checks are merged into one
  frame with a leading `block` column via `blocks(...)`.
- Every frame is also written to `OUT_DIR` as CSV by `save()`.
- QA checks register through `gate()` and surface as PASS/FAIL rows inside the
  cell's own output, rather than raising and hiding the table.
- Heavy cells are flag-gated. Run `PROFILE_MONTHS` first, then widen.

### Run order

`D0 → D1..D8` (deposit truth) → `A1` (avg_monthly_bal_1) → `M1..M3` (panel,
closure labels, lead_gap) → `P1..P4` (payments) → `J1..J2` (join) → `Z`.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import os, re, json, time
import numpy as np
import pandas as pd

# --- environment compat ------------------------------------------------------
# Cloudera SPARK3's PySpark is older than the NumPy in conda env `new_default`;
# toPandas() on an empty/Arrow path reaches for aliases removed in NumPy 2.
for _alias, _type in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
                      ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _type)
# PySpark 3.3 createDataFrame(pandas) calls .iteritems, removed in pandas 2.0.
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from pyspark.sql import SparkSession, functions as F, Window as W

CONFIG = dict(
    # --- sources -------------------------------------------------------------
    TXN_TABLE   = "dsihd01p_dsi.neo4j_payments",
    DEP_TABLE   = "<db>.<deposit_daily>",                      # <-- FILL THIS IN
    DEDUP_TABLE = "bdahd01p_dlcdi1_cdi_tm.pkg_dedup_5c6c",     # 5C/6C rows to drop

    WORK_DIR = "hdfs:///user/pk36814/pkg_attrition_v5/work",
    OUT_DIR  = "../eda/attrition_v5",

    # --- window --------------------------------------------------------------
    MONTH_MIN = "2024-01",
    MONTH_MAX = "2026-07",

    # Heavy transaction profiling runs on these months first. Set to None to use
    # the full window — at ~400M txn rows/month that is ~12B rows, so don't do it
    # by accident.
    PROFILE_MONTHS = ["2024-06", "2025-06", "2026-06"],

    # --- stage control -------------------------------------------------------
    RUN_TXN_TAXONOMY = False,   # rail/category/cpty-key profiling: settled in v3
    RUN_HEAVY_DEP    = True,    # full-shuffle deposit grain checks (D4, D5)
    APPLY_DEDUP      = True,    # 5C/6C dedup, at the START of the pipeline
    DROP_CARD        = True,    # 8B/8C debit card — TM focus is businesses
    REBUILD          = False,   # ignore cached parquet in WORK_DIR

    # Profile deposits on a hash subset of accounts. 100 = everything.
    ACCT_SAMPLE_PCT = 100,

    # --- domain --------------------------------------------------------------
    BALANCE_FLOOR     = 25_000,   # brief §6.2 small-balance exclusion
    CURRENT_RULE_DROP = 0.30,     # incumbent rule: trailing-3 vs prior-6
    T0_MEDIAN_FRAC    = 0.95,     # brief §6.1 top-of-slide threshold
    PERSISTENCE_MONTHS = 3,
    BURN_IN_MONTHS     = 12,
    SILENCE_MONTHS     = 3,
    HLL_RSD            = 0.01,
)

# Column maps. One place to fix if the source names differ.
T = dict(
    txn_id="trans_id", date="trans_dt", amount="trans_amt",
    rail="payment_rail", category="category", src_syst="src_syst",
    pays_id="mdm_id_pays", pays_name="customer_name_pays", pays_acct="pnc_dep_acct_pays",
    recv_id="mdm_id_receives", recv_name="customer_name_receives", recv_acct="pnc_dep_acct_receives",
    cpty_acct="unq_cpty_acct_id", cpty_hash="unq_cpty_id", cpty_name="cpty_name",
    cpty_fi="cpty_fin_entity_name", cpty_type="cpty_type",
)

DEPC = dict(
    acct="acct_full_acct_id", load_dt="edw_tda_load_dt", balance="balance",
    avg_bal_1="avg_monthly_bal_1", status="acct_status", status_desc="acct_status_desc",
    family="deposit_family", cust="cust_pwr_id", cust_name="cust_name",
    opened="opened_dt", closed="closed_dt",
)
DEP_REQUIRED = ["acct", "load_dt", "balance", "status", "family"]

# Counterparty key rule from the v3 finding: unq_cpty_acct_id is null on
# 7B/8B/8C/9B and is only a routing number on 3B/3C.
CPTY_HASH_CATS = ["3B", "3C", "7B", "9B"]
CARD_CATS      = ["8B", "8C"]

# Confirmed taxonomy — assertion, not discovery.
KNOWN = dict(
    RAILS = {"WIRE", "ACH", "CHECK", "RTP_PRT", "PCARD", "DEBIT_CARD_SIGNATURE", "RTP_P2P"},
    DIRECTION_LETTER = {"A": "INTERNAL", "B": "OUTBOUND", "C": "INBOUND"},
)

# Tax/treasury names that carry the PAYER's own name in the counterparty field —
# the same-name false-positive mechanism found in v3.
TAX_FP_PATTERN = (r"(TREASURY|IRS|EFTPS|TAXPAYOR|TAXPAYER|DEPT\s*REVENUE|"
                  r"FRANCHISE\s*TAX|DEPT\s*TAXATION|WEBFILE|BWC)")
# Counterparty names that are themselves financial institutions. This is what the
# brief's `fi_destination_flag` actually needs — see the note above cell P3.
FI_NAME_PATTERN = (r"\b(BANK|BANCORP|BANCSHARES|CREDIT UNION|FEDERAL CREDIT|SAVINGS|"
                   r"TRUST CO|N A|NATIONAL ASSOCIATION|FINANCIAL|FSB|BROKERAGE|"
                   r"SECURITIES|FIDELITY|SCHWAB|VANGUARD)\b")

In [ ]:
# ============================================================================
# SPARK SESSION & HELPERS
# ============================================================================
spark = (SparkSession.builder
         .appName("pkg_attrition_source_eda_v5")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .getOrCreate())

for _k, _v in {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.shuffle.partitions": "2000",
    "spark.sql.parquet.filterPushdown": "true",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.execution.arrow.pyspark.fallback.enabled": "true",
    "spark.sql.autoBroadcastJoinThreshold": str(64 * 1024 * 1024),
    "spark.sql.sources.partitionOverwriteMode": "dynamic",
    "spark.sql.legacy.timeParserPolicy": "CORRECTED",
}.items():
    spark.conf.set(_k, _v)

OUT, WORK = CONFIG["OUT_DIR"], CONFIG["WORK_DIR"]
os.makedirs(OUT, exist_ok=True)

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

R, TIMINGS, GATES = {}, [], []


# --- display: one frame per cell ---------------------------------------------
def save(pdf, name, note=""):
    """Write a summary frame to CSV and return it so the cell renders it inline."""
    if len(pdf) > 5000:
        raise ValueError(f"{name}: {len(pdf)} rows is not a summary")
    pdf.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)
    R[name] = pdf
    return pdf


def blocks(*pairs):
    """Concatenate (label, frame) pairs into ONE frame with a leading `block` column.
    Schemas need not match — this exists so a cell has a single screenshotable output."""
    out = []
    for label, frame in pairs:
        f = frame.copy()
        f.insert(0, "block", label)
        out.append(f)
    return pd.concat(out, ignore_index=True, sort=False)


def kv(*rows):
    """[(metric, value), ...] -> two-column frame."""
    return pd.DataFrame(rows, columns=["metric", "value"])


def agg1(sdf, **aggs):
    """Run ONE aggregation pass and return it long-format. Keeps actions to one
    per cell instead of one per metric."""
    row = sdf.agg(*[v.alias(k) for k, v in aggs.items()]).toPandas().iloc[0]
    return pd.DataFrame({"metric": list(aggs.keys()),
                         "value": [row[k] for k in aggs.keys()]})


def P(sdf, n=1000):
    """Collect to pandas with a hard cap. Never .show()."""
    return sdf.limit(n).toPandas()


def gate(label, ok, detail=""):
    GATES.append({"gate": label, "status": "PASS" if ok else "FAIL", "detail": str(detail)})
    return bool(ok)


def gates_frame(last=None):
    g = pd.DataFrame(GATES)
    return g.tail(last) if last else g


class timed:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a):
        TIMINGS.append({"stage": self.label, "seconds": round(time.time() - self.t, 1)})


# --- column / value helpers ---------------------------------------------------
def d(col):
    """Parse to date whether the source column is date or 'yyyy-MM-dd' string."""
    return F.to_date(F.col(col)) if isinstance(col, str) else F.to_date(col)


def present(c):
    col = F.col(c) if isinstance(c, str) else c
    return col.isNotNull() & (F.trim(col.cast("string")) != "")


def ndv(c, rsd=None):
    return F.approx_count_distinct(c, rsd or CONFIG["HLL_RSD"])


def norm_name(col):
    c = F.upper(F.trim(col))
    c = F.regexp_replace(c, r"[^A-Z0-9 ]", " ")
    c = F.regexp_replace(c, r"\b(LLC|L L C|INC|INCORPORATED|CORP|CORPORATION|CO|LP|LLP|"
                            r"PLLC|PC|LTD|LIMITED|DBA|THE)\b", " ")
    return F.trim(F.regexp_replace(c, r"\s+", " "))


def mo_of(col):
    """'yyyy-MM' month key from a date or date-string column."""
    return F.date_format(d(col), "yyyy-MM")


def mo_idx(col):
    """Integer month index off MONTH_MIN. Used for rangeBetween windows, which
    tolerate gaps — rowsBetween would silently mis-window a sparse account."""
    anchor = F.lit(CONFIG["MONTH_MIN"] + "-01").cast("date")
    return F.months_between(F.to_date(F.concat(F.col(col), F.lit("-01"))), anchor).cast("int")


def month_range(lo, hi):
    out, y, m = [], int(lo[:4]), int(lo[5:7])
    while f"{y:04d}-{m:02d}" <= hi:
        out.append(f"{y:04d}-{m:02d}")
        m += 1
        if m == 13:
            y, m = y + 1, 1
    return out


MONTHS = month_range(CONFIG["MONTH_MIN"], CONFIG["MONTH_MAX"])
DT_START, DT_END = MONTHS[0] + "-01", None
_y, _m = int(MONTHS[-1][:4]), int(MONTHS[-1][5:7])
DT_END = (pd.Timestamp(year=_y, month=_m, day=1) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

PROFILE_MONTHS = CONFIG["PROFILE_MONTHS"] or MONTHS


def acct_sample(sdf, col):
    """Deterministic hash subset of accounts for heavy profiling cells."""
    pct = CONFIG["ACCT_SAMPLE_PCT"]
    return sdf if pct >= 100 else sdf.where((F.crc32(F.col(col).cast("string")) % 100) < pct)


def path(n): return f"{WORK}/{n}"


def has_parquet(n):
    try:
        spark.read.parquet(path(n)).limit(1).count()
        return True
    except Exception:
        return False


def cache_parquet(sdf, name, partition_by=None):
    """Materialize once, re-read cheaply on later runs."""
    if CONFIG["REBUILD"] or not has_parquet(name):
        w = sdf.write.mode("overwrite")
        if partition_by:
            w = w.partitionBy(partition_by)
        w.parquet(path(name))
    return spark.read.parquet(path(name))


save(kv(("spark_version", spark.version),
        ("pandas", pd.__version__), ("numpy", np.__version__),
        ("months", f"{MONTHS[0]} .. {MONTHS[-1]} ({len(MONTHS)})"),
        ("date_filter", f"{DT_START} .. {DT_END}"),
        ("profile_months", ", ".join(PROFILE_MONTHS)),
        ("txn_table", CONFIG["TXN_TABLE"]), ("dep_table", CONFIG["DEP_TABLE"]),
        ("out_dir", OUT), ("work_dir", WORK),
        ("acct_sample_pct", CONFIG["ACCT_SAMPLE_PCT"]),
        ("flags", f"taxonomy={CONFIG['RUN_TXN_TAXONOMY']} heavy_dep={CONFIG['RUN_HEAVY_DEP']} "
                  f"dedup={CONFIG['APPLY_DEDUP']} drop_card={CONFIG['DROP_CARD']}")),
     "env")

## D0 · Deposit schema map

Confirms every column this notebook depends on exists and reports its type,
plus everything in the table we are not using. If a name differs, fix `DEPC`
in CONFIG — nothing downstream refers to a literal column name.

In [ ]:
dep_raw = spark.table(CONFIG["DEP_TABLE"])
_dtypes = pd.DataFrame(dep_raw.dtypes, columns=["column", "dtype"])
_cols = set(dep_raw.columns)

_mapped = pd.DataFrame([{"role": k, "column": v, "required": k in DEP_REQUIRED,
                         "present": v in _cols} for k, v in DEPC.items()])
_mapped = _mapped.merge(_dtypes, on="column", how="left")

_unmapped = pd.DataFrame({"column": sorted(_cols - set(DEPC.values()))}) \
              .merge(_dtypes, on="column", how="left")

_missing_req = _mapped[(_mapped["required"]) & (~_mapped["present"])]["role"].tolist()
gate("D0 required deposit columns", not _missing_req, f"missing={_missing_req}")

ACCT, LOAD, BAL = DEPC["acct"], DEPC["load_dt"], DEPC["balance"]

save(blocks(("mapped", _mapped),
            ("unmapped", _unmapped),
            ("gates", gates_frame(1))), "d0_schema_map")

## D1 · Window, headline grain, monthly coverage

One pass. Establishes whether the claim "one row per account per business day"
holds, and whether the account population is stable month to month or growing
(which would mean the table is an accumulating snapshot, not a panel).

In [ ]:
with timed("D1 deposit grain"):
    dep = (dep_raw
           .withColumn("load_d", d(LOAD))
           .where((F.col("load_d") >= F.lit(DT_START).cast("date")) &
                  (F.col("load_d") <= F.lit(DT_END).cast("date")))
           .withColumn("mo", F.date_format("load_d", "yyyy-MM")))
    dep = acct_sample(dep, ACCT)

    _head_aggs = dict(rows=F.count(F.lit(1)), accounts=ndv(ACCT))
    if DEPC["cust"] in _cols:
        _head_aggs["customers"] = ndv(DEPC["cust"])
    _head = agg1(dep, **_head_aggs,
                 load_dates=F.countDistinct("load_d"),
                 months=F.countDistinct("mo"),
                 first_dt=F.min("load_d"), last_dt=F.max("load_d"),
                 families=F.countDistinct(DEPC["family"]),
                 statuses=F.countDistinct(DEPC["status"]))

    _monthly = (dep.groupBy("mo")
                .agg(F.count(F.lit(1)).alias("rows"),
                     ndv(ACCT).alias("accounts"),
                     F.countDistinct("load_d").alias("load_dates"),
                     F.avg(BAL).alias("avg_bal"),
                     F.sum(F.col(BAL)).alias("sum_bal"))
                .orderBy("mo").toPandas())
    _monthly["rows_per_acct"] = _monthly["rows"] / _monthly["accounts"]

    # A month with fewer load dates than business days is a feed gap, not a
    # closure pattern — worth knowing before any silence metric is defined.
    _bd = {m: int(np.busday_count(f"{m}-01",
                   (pd.Timestamp(m + "-01") + pd.offsets.MonthEnd(1) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")))
           for m in _monthly["mo"]}
    _monthly["business_days"] = _monthly["mo"].map(_bd)
    _monthly["date_gap"] = _monthly["business_days"] - _monthly["load_dates"]

gate("D1 all months present", len(_monthly) == len(MONTHS),
     f"{len(_monthly)} of {len(MONTHS)}")
gate("D1 no month missing >2 load dates", (_monthly["date_gap"] <= 2).all(),
     f"max gap {_monthly['date_gap'].max()}")

save(blocks(("headline", _head), ("monthly", _monthly), ("gates", gates_frame(2))),
     "d1_grain")

## D2 · Column population and cardinality

Single pass over the mapped columns. `ndv` is HyperLogLog at 1% relative error,
so treat counts as approximate — exact distincts on a table this size are not
worth the shuffle at profiling stage.

In [ ]:
with timed("D2 column profile"):
    _prof_cols = [v for v in DEPC.values() if v in _cols]
    _aggs = {"__n": F.count(F.lit(1))}
    for c in _prof_cols:
        _aggs[f"nn__{c}"] = F.sum(present(c).cast("long"))
        _aggs[f"dv__{c}"] = ndv(c)
    _row = dep.agg(*[v.alias(k) for k, v in _aggs.items()]).toPandas().iloc[0]

    _n = int(_row["__n"])
    _cp = pd.DataFrame({
        "column": _prof_cols,
        "role": [k for k in DEPC if DEPC[k] in _prof_cols],
        "non_null": [int(_row[f"nn__{c}"]) for c in _prof_cols],
        "approx_distinct": [int(_row[f"dv__{c}"]) for c in _prof_cols],
    })
    _cp["null_pct"] = (1 - _cp["non_null"] / _n) * 100
    _cp = _cp.sort_values("null_pct", ascending=False)

save(blocks(("rows", kv(("rows_profiled", _n))), ("columns", _cp)), "d2_columns")

## D3 · `acct_status` × `acct_status_desc` × `deposit_family`

The point of this cell is the **code dictionary**: which statuses exist, what
they mean, and which account types they occur on. `'C'` is the documented
closed code, but any other code that behaves like a terminal state (dormant,
escheat, charged-off) matters just as much for the label, and only the
description column can tell us which those are.

In [ ]:
with timed("D3 status taxonomy"):
    _st = (dep.groupBy(DEPC["status"], DEPC["status_desc"], DEPC["family"])
           .agg(F.count(F.lit(1)).alias("rows"),
                ndv(ACCT).alias("accounts"),
                F.sum(present(DEPC["closed"]).cast("long")).alias("rows_with_closed_dt"),
                F.avg(BAL).alias("avg_bal"),
                F.sum((F.col(BAL) == 0).cast("long")).alias("rows_zero_bal"),
                F.min("load_d").alias("first_dt"), F.max("load_d").alias("last_dt"))
           .orderBy(F.desc("rows")).limit(300).toPandas())
    _st["closed_dt_pct"] = _st["rows_with_closed_dt"] / _st["rows"] * 100
    _st["zero_bal_pct"] = _st["rows_zero_bal"] / _st["rows"] * 100

    _st_roll = (_st.groupby([DEPC["status"], DEPC["status_desc"]], dropna=False)
                .agg(rows=("rows", "sum"), accounts=("accounts", "sum"),
                     families=(DEPC["family"], "nunique"),
                     closed_dt_pct=("closed_dt_pct", "mean"),
                     zero_bal_pct=("zero_bal_pct", "mean"))
                .reset_index().sort_values("rows", ascending=False))

save(blocks(("by_status", _st_roll), ("by_status_family", _st)), "d3_status_taxonomy")

## D4 · Is it really one row per account per business day?

Three failure modes worth separating: duplicate account-days (join hazard),
accounts absent on days they should be present (the "silence" signal is only
meaningful if absence is rare and systematic), and accounts that appear
mid-window (new opens vs. backfill).

In [ ]:
if CONFIG["RUN_HEAVY_DEP"]:
    with timed("D4 panel completeness"):
        _ad = dep.groupBy(ACCT, "load_d").agg(F.count(F.lit(1)).alias("n"))
        _dup = agg1(_ad,
                    acct_days=F.count(F.lit(1)),
                    dup_acct_days=F.sum((F.col("n") > 1).cast("long")),
                    max_rows_per_acct_day=F.max("n"))

        # Per account-month: days present vs load dates available that month.
        _mo_dates = dep.groupBy("mo").agg(F.countDistinct("load_d").alias("mo_load_dates"))
        _am = (dep.groupBy(ACCT, "mo").agg(F.countDistinct("load_d").alias("days_present"))
               .join(_mo_dates, "mo"))
        _am = _am.withColumn("complete", F.col("days_present") == F.col("mo_load_dates"))
        _cov = agg1(_am,
                    account_months=F.count(F.lit(1)),
                    complete_months=F.sum(F.col("complete").cast("long")),
                    partial_months=F.sum((~F.col("complete")).cast("long")),
                    median_days_present=F.expr("percentile_approx(days_present, 0.5, 1000)"))

        _dist = (_am.withColumn("pct_present",
                                F.round(F.col("days_present") / F.col("mo_load_dates") * 10) * 10)
                 .groupBy("pct_present").agg(F.count(F.lit(1)).alias("account_months"))
                 .orderBy("pct_present").toPandas())

    gate("D4 no duplicate account-days",
         int(_dup.loc[_dup.metric == "dup_acct_days", "value"].iloc[0]) == 0,
         "duplicates break every downstream join on (acct, date)")

    _out = blocks(("duplicates", _dup), ("coverage", _cov),
                  ("days_present_hist", _dist), ("gates", gates_frame(1)))
else:
    _out = kv(("skipped", "RUN_HEAVY_DEP=False"))
save(_out, "d4_panel_completeness")

## D5 · Account dimension, and what happens after `closed_dt`

**This answers the open question directly.** If rows continue after `closed_dt`,
then "the record stopped appearing" is not a usable closure signal and the
panel stays balanced through closure — which is good news, because it means a
closed account still contributes months to the feature window. If rows stop,
then account disappearance *is* the label and the panel is unbalanced by
construction, which changes how every trailing window has to be computed.

In [ ]:
with timed("D5 account dim + post-closure"):
    acct_dim = (dep.groupBy(ACCT).agg(
        F.min("load_d").alias("first_load"),
        F.max("load_d").alias("last_load"),
        F.count(F.lit(1)).alias("n_rows"),
        F.countDistinct("mo").alias("n_months"),
        F.sum((F.col(DEPC["status"]) == "C").cast("long")).alias("n_rows_status_c"),
        F.countDistinct(DEPC["status"]).alias("n_distinct_status"),
        F.min(d(DEPC["closed"])).alias("first_closed_d"),
        F.max(d(DEPC["closed"])).alias("last_closed_d"),
        F.countDistinct(d(DEPC["closed"])).alias("n_distinct_closed_d"),
        F.max(F.struct(F.col("load_d").alias("d"),
                       F.col(DEPC["status"]).alias("status"),
                       F.col(BAL).alias("balance"),
                       d(DEPC["opened"]).alias("opened_d"),
                       d(DEPC["closed"]).alias("closed_d"),
                       F.col(DEPC["family"]).alias("family"),
                       F.col(DEPC["cust"]).alias("cust"))).alias("last"))
        .select("*", F.col("last.status").alias("last_status"),
                F.col("last.balance").alias("last_balance"),
                F.col("last.opened_d").alias("opened_d"),
                F.col("last.closed_d").alias("closed_d"),
                F.col("last.family").alias("family"),
                F.col("last.cust").alias("cust"))
        .drop("last"))
    acct_dim = cache_parquet(acct_dim, "acct_dim").cache()

    _panel_end = dep.agg(F.max("load_d")).collect()[0][0]

    _ad_sum = agg1(acct_dim,
                   accounts=F.count(F.lit(1)),
                   with_closed_dt=F.sum(present("closed_d").cast("long")),
                   with_status_c_ever=F.sum((F.col("n_rows_status_c") > 0).cast("long")),
                   last_status_c=F.sum((F.col("last_status") == "C").cast("long")),
                   multi_status=F.sum((F.col("n_distinct_status") > 1).cast("long")),
                   multi_closed_dt=F.sum((F.col("n_distinct_closed_d") > 1).cast("long")),
                   stops_before_panel_end=F.sum(
                       (F.col("last_load") < F.date_sub(F.lit(_panel_end), 7)).cast("long")))

    # Rows dated after the account's own closed_dt.
    _after = (dep.join(acct_dim.select(F.col(ACCT), F.col("closed_d")).where(present("closed_d")),
                       on=ACCT, how="inner")
              .where(F.col("load_d") > F.col("closed_d")))
    _after_sum = agg1(_after,
                      rows_after_closed=F.count(F.lit(1)),
                      accounts_with_rows_after=ndv(ACCT),
                      rows_status_c=F.sum((F.col(DEPC["status"]) == "C").cast("long")),
                      rows_nonzero_bal=F.sum((F.abs(F.col(BAL)) > 0.005).cast("long")),
                      median_days_after=F.expr("percentile_approx(datediff(load_d, closed_d), 0.5, 1000)"),
                      max_days_after=F.max(F.datediff("load_d", "closed_d")))

    # How long an account keeps reporting past closure.
    _tail = (acct_dim.where(present("closed_d"))
             .withColumn("days_after_close", F.datediff("last_load", "closed_d"))
             .withColumn("bucket", F.when(F.col("days_after_close") < 0, "still_open_after_close_dt")
                         .when(F.col("days_after_close") == 0, "0_last_row_is_close_day")
                         .when(F.col("days_after_close") <= 5, "1-5d")
                         .when(F.col("days_after_close") <= 31, "6-31d")
                         .when(F.col("days_after_close") <= 93, "32-93d")
                         .otherwise("94d+"))
             .groupBy("bucket").agg(F.count(F.lit(1)).alias("accounts"),
                                    F.avg("last_balance").alias("avg_last_bal"))
             .orderBy("bucket").toPandas())

save(blocks(("accounts", _ad_sum), ("rows_after_closed_dt", _after_sum),
            ("tail_after_close", _tail)), "d5_closure_behaviour")

## D6 · Status transitions

Does an account ever leave `'C'`? A reopen means `'C'` is a state, not an
absorbing terminal event, and a label built on "last status" would silently
disagree with one built on "ever closed".

In [ ]:
with timed("D6 status transitions"):
    _am_st = (dep.groupBy(ACCT, "mo")
              .agg(F.max(F.struct(F.col("load_d").alias("d"),
                                  F.col(DEPC["status"]).alias("st"))).alias("l"))
              .select(F.col(ACCT), F.col("mo"), F.col("l.st").alias("status")))
    _w = W.partitionBy(ACCT).orderBy("mo")
    _tr = (_am_st.withColumn("prev", F.lag("status").over(_w))
           .where(F.col("prev").isNotNull() & (F.col("prev") != F.col("status"))))
    _mat = (_tr.groupBy("prev", "status").agg(F.count(F.lit(1)).alias("transitions"),
                                              ndv(ACCT).alias("accounts"))
            .orderBy(F.desc("transitions")).limit(100).toPandas())
    _reopen = agg1(_tr.where(F.col("prev") == "C"),
                   reopen_transitions=F.count(F.lit(1)),
                   reopen_accounts=ndv(ACCT))

gate("D6 'C' is absorbing",
     int(_reopen.loc[_reopen.metric == "reopen_transitions", "value"].iloc[0]) == 0,
     "non-zero means last-status and ever-closed labels will disagree")

save(blocks(("transition_matrix", _mat), ("reopen", _reopen), ("gates", gates_frame(1))),
     "d6_status_transitions")

## D7 · Maturity closures vs. decisions

A CD that closes on its maturity date is not attrition. If tenure at closure
bunches at 6/12/24/36/60 months for a family, that family's closures are
contractual and have to be excluded from the label — otherwise the model
learns the term sheet.

In [ ]:
with timed("D7 maturity check"):
    _cl = (acct_dim.where(present("closed_d") & present("opened_d"))
           .withColumn("tenure_m", F.round(F.months_between("closed_d", "opened_d")).cast("int")))

    _by_fam = (_cl.groupBy("family")
               .agg(F.count(F.lit(1)).alias("closed_accounts"),
                    F.expr("percentile_approx(tenure_m, 0.5, 1000)").alias("median_tenure_m"),
                    F.sum(F.col("tenure_m").isin([6, 12, 18, 24, 36, 48, 60]).cast("long")).alias("round_tenor_n"),
                    F.avg("last_balance").alias("avg_last_bal"))
               .orderBy(F.desc("closed_accounts")).limit(60).toPandas())
    _by_fam["round_tenor_pct"] = _by_fam["round_tenor_n"] / _by_fam["closed_accounts"] * 100

    # Spike detection: the top tenure values per family. Bunching shows up here
    # even if the round-tenor list above misses the actual tenor.
    _spike = (_cl.groupBy("family", "tenure_m").agg(F.count(F.lit(1)).alias("n"))
              .withColumn("rk", F.row_number().over(
                  W.partitionBy("family").orderBy(F.desc("n"))))
              .where(F.col("rk") <= 3).orderBy("family", "rk").limit(200).toPandas())

save(blocks(("by_family", _by_fam), ("top_tenures", _spike)), "d7_maturity")

## D8 · Balance distribution

Negative balances are expected and legitimate here. What matters is how common
they are, because every ratio metric in the brief (`net_flow_ratio`,
`balance_to_flow_ratio`) is undefined or sign-flipped across zero.

In [ ]:
with timed("D8 balance distribution"):
    _q = (dep.agg(F.expr(f"percentile_approx({BAL}, array(0.01,0.05,0.25,0.5,0.75,0.95,0.99), 1000)").alias("q"))
          .toPandas().iloc[0]["q"])
    _qf = pd.DataFrame({"quantile": ["p01", "p05", "p25", "p50", "p75", "p95", "p99"],
                        "balance": list(_q)})

    _sign = (dep.groupBy(DEPC["family"])
             .agg(F.count(F.lit(1)).alias("rows"),
                  F.sum((F.col(BAL) < 0).cast("long")).alias("neg_rows"),
                  F.sum((F.col(BAL) == 0).cast("long")).alias("zero_rows"),
                  F.min(BAL).alias("min_bal"), F.max(BAL).alias("max_bal"))
             .orderBy(F.desc("rows")).limit(60).toPandas())
    _sign["neg_pct"] = _sign["neg_rows"] / _sign["rows"] * 100
    _sign["zero_pct"] = _sign["zero_rows"] / _sign["rows"] * 100

save(blocks(("quantiles", _qf), ("by_family", _sign)), "d8_balance")

## A1 · What is `avg_monthly_bal_1`?

Three hypotheses, one discriminating test:

| If it is… | Then within a month it is… | And it equals… |
|---|---|---|
| Prior complete month | **constant** across all days | prior month's mean of `balance` |
| Month-to-date | rising/falling day to day | running mean of `balance` within the month |
| Current full month | constant | this month's mean of `balance` |

The constancy test alone separates MTD from the other two; the equality tests
separate prior-month from current-month. Runs on a 1% account sample — this is
a semantics question, not an estimation one.

In [ ]:
_A1_PCT = 1
if DEPC["avg_bal_1"] in _cols:
    with timed("A1 avg_monthly_bal_1 semantics"):
        AB = DEPC["avg_bal_1"]
        _s = dep.where((F.crc32(F.col(ACCT).cast("string")) % 100) < _A1_PCT)

        _const = (_s.groupBy(ACCT, "mo")
                  .agg(F.countDistinct(F.round(F.col(AB), 2)).alias("n_vals"),
                       F.count(F.lit(1)).alias("n_days")))
        _const_sum = agg1(_const.where(F.col("n_days") >= 5),
                          account_months=F.count(F.lit(1)),
                          constant_within_month=F.sum((F.col("n_vals") <= 1).cast("long")),
                          varies_within_month=F.sum((F.col("n_vals") > 1).cast("long")))

        _mo_stats = _s.groupBy(ACCT, "mo").agg(F.avg(BAL).alias("mo_mean"))
        _prior = _mo_stats.select(
            F.col(ACCT),
            F.date_format(F.add_months(F.to_date(F.concat(F.col("mo"), F.lit("-01"))), 1),
                          "yyyy-MM").alias("mo"),
            F.col("mo_mean").alias("prior_mo_mean"))

        _wm = W.partitionBy(ACCT, "mo").orderBy("load_d").rowsBetween(W.unboundedPreceding, 0)
        _t = (_s.withColumn("mtd_mean", F.avg(BAL).over(_wm))
              .join(_mo_stats, [ACCT, "mo"], "left")
              .join(_prior, [ACCT, "mo"], "left")
              .withColumn("dom", F.dayofmonth("load_d")))

        def _match(a, b):
            return (F.abs(F.col(a) - F.col(b)) <=
                    0.001 * F.greatest(F.abs(F.col(b)), F.lit(1.0))).cast("long")

        _hyp = agg1(_t.where(present(AB)),
                    rows_tested=F.count(F.lit(1)),
                    match_prior_month=F.avg(_match(AB, "prior_mo_mean")),
                    match_current_month_full=F.avg(_match(AB, "mo_mean")),
                    match_month_to_date=F.avg(_match(AB, "mtd_mean")))

        # If MTD, the match to prior-month collapses after day 1 while the match
        # to the running mean holds flat. If prior-month, the reverse.
        _dom = (_t.where(present(AB))
                .withColumn("dom_bucket", F.when(F.col("dom") <= 3, "01-03")
                            .when(F.col("dom") <= 10, "04-10")
                            .when(F.col("dom") <= 20, "11-20").otherwise("21-31"))
                .groupBy("dom_bucket")
                .agg(F.count(F.lit(1)).alias("rows"),
                     F.avg(_match(AB, "prior_mo_mean")).alias("match_prior"),
                     F.avg(_match(AB, "mo_mean")).alias("match_current"),
                     F.avg(_match(AB, "mtd_mean")).alias("match_mtd"))
                .orderBy("dom_bucket").toPandas())

    _out = blocks(("constancy", _const_sum), ("hypotheses", _hyp), ("by_day_of_month", _dom))
else:
    _out = kv(("skipped", f"{DEPC['avg_bal_1']} not in table"))
save(_out, "a1_avg_monthly_bal")

## M1 · Monthly deposit panel

Account-month grain. Keeps **both** balance conventions — month-end and
monthly-average — because the brief's rule is written on an average while the
benchmark curve it is compared against is an average DDA balance, and the two
behave differently for accounts with intra-month swings. `days_present` stays
on the panel so a partial month can be excluded rather than averaged as if
complete.

In [ ]:
with timed("M1 monthly panel"):
    panel = (dep.groupBy(ACCT, "mo").agg(
        F.count(F.lit(1)).alias("days_present"),
        F.avg(BAL).alias("avg_bal"),
        F.min(BAL).alias("min_bal"),
        F.max(BAL).alias("max_bal"),
        F.expr(f"percentile_approx({BAL}, 0.5, 1000)").alias("med_bal"),
        F.sum((F.col(DEPC["status"]) == "C").cast("long")).alias("days_status_c"),
        F.max(F.struct(F.col("load_d").alias("d"),
                       F.col(BAL).alias("bal"),
                       F.col(DEPC["status"]).alias("status"),
                       F.col(DEPC["family"]).alias("family"),
                       F.col(DEPC["cust"]).alias("cust"),
                       d(DEPC["closed"]).alias("closed_d"),
                       d(DEPC["opened"]).alias("opened_d"))).alias("eo"))
        .select("*",
                F.col("eo.d").alias("eom_dt"), F.col("eo.bal").alias("eom_bal"),
                F.col("eo.status").alias("eom_status"), F.col("eo.family").alias("family"),
                F.col("eo.cust").alias("cust"), F.col("eo.closed_d").alias("closed_d"),
                F.col("eo.opened_d").alias("opened_d"))
        .drop("eo")
        .withColumn("mi", mo_idx("mo")))
    panel = cache_parquet(panel, "dep_panel_monthly", partition_by="mo")

    _p_sum = agg1(panel,
                  account_months=F.count(F.lit(1)),
                  accounts=ndv(ACCT),
                  customers=ndv("cust"),
                  months=F.countDistinct("mo"),
                  months_status_c=F.sum((F.col("eom_status") == "C").cast("long")),
                  months_partial=F.sum((F.col("days_present") < 15).cast("long")),
                  avg_bal_vs_eom_corr=F.corr("avg_bal", "eom_bal"))

    _p_mo = (panel.groupBy("mo").agg(
        F.count(F.lit(1)).alias("accounts"),
        F.sum((F.col("eom_status") == "C").cast("long")).alias("status_c"),
        F.avg("avg_bal").alias("mean_avg_bal"),
        F.expr("percentile_approx(avg_bal, 0.5, 1000)").alias("median_avg_bal"))
        .orderBy("mo").toPandas())

save(blocks(("panel", _p_sum), ("by_month", _p_mo)), "m1_panel")

## M2 · Five closure definitions, scored against the real flag

The brief assumed no closure label existed and used a balance-decline proxy.
We have a label, so the question inverts: **which definition of "closed" should
be the target, and how badly does the proxy approximate it?**

| Label | Definition |
|---|---|
| `status_c` | first month whose month-end `acct_status` is `'C'` |
| `closed_dt` | month of the populated `closed_dt` |
| `record_stop` | month after the last month the account appears, if that is before panel end |
| `zero_run` | first month starting a run of 3 months at ~zero balance |
| `decline30` | the incumbent rule firing — the brief's proxy, not a closure at all |

Read the pairwise output as: how often they co-occur, and when both fire, how
many months apart. A proxy that fires four months *after* the real flag is
worse than useless as an early warning.

In [ ]:
with timed("M2 closure definitions"):
    _w = W.partitionBy(ACCT).orderBy("mi")
    _w3 = _w.rangeBetween(-2, 0)
    _w6p = _w.rangeBetween(-8, -3)
    _w12 = _w.rangeBetween(-12, -1)
    _wfwd3 = _w.rangeBetween(0, 2)

    _pn = (panel
           .withColumn("avg3", F.avg("avg_bal").over(_w3))
           .withColumn("avg6_prior", F.avg("avg_bal").over(_w6p))
           .withColumn("n_prior", F.count("avg_bal").over(_w12))
           .withColumn("zero_fwd3", F.sum((F.abs(F.col("avg_bal")) < 1.0).cast("int")).over(_wfwd3)))

    try:
        _pn = _pn.withColumn("med12", F.expr("percentile_approx(avg_bal, 0.5, 100)").over(_w12))
        _pn.limit(1).count()
        _median_mode = "percentile_approx_window"
    except Exception as e:
        _pn = _pn.withColumn("med12", F.avg("avg_bal").over(_w12))
        _median_mode = f"mean_fallback ({type(e).__name__})"
    gate("M2 trailing-12 median", _median_mode.startswith("percentile"), _median_mode)

    _pn = _pn.withColumn(
        "rule30",
        (F.col("n_prior") >= CONFIG["BURN_IN_MONTHS"]) &
        (F.col("avg6_prior") >= CONFIG["BALANCE_FLOOR"]) &
        (F.col("avg3") < (1 - CONFIG["CURRENT_RULE_DROP"]) * F.col("avg6_prior")))
    _pn = cache_parquet(_pn, "dep_panel_rules", partition_by="mo")

    _panel_end_mi = panel.agg(F.max("mi")).collect()[0][0]

    _lab = (_pn.groupBy(ACCT).agg(
        F.min(F.when(F.col("eom_status") == "C", F.col("mi"))).alias("mi_status_c"),
        F.min(F.when(present("closed_d"),
                     F.months_between(F.trunc(F.col("closed_d"), "MM"),
                                      F.lit(CONFIG["MONTH_MIN"] + "-01").cast("date")).cast("int"))).alias("mi_closed_dt"),
        F.max("mi").alias("mi_last"),
        F.min(F.when(F.col("zero_fwd3") >= 3, F.col("mi"))).alias("mi_zero_run"),
        F.min(F.when(F.col("rule30"), F.col("mi"))).alias("mi_decline30"))
        .withColumn("mi_record_stop",
                    F.when(F.col("mi_last") < _panel_end_mi, F.col("mi_last") + 1)))

    _defs = ["mi_status_c", "mi_closed_dt", "mi_record_stop", "mi_zero_run", "mi_decline30"]
    _counts = agg1(_lab, accounts=F.count(F.lit(1)),
                   **{dfn.replace("mi_", "n_"): F.sum(F.col(dfn).isNotNull().cast("long"))
                      for dfn in _defs})

    _pairs, _aggs = [], {}
    for i, a in enumerate(_defs):
        for b in _defs[i + 1:]:
            k = f"{a[3:]}__{b[3:]}"
            _pairs.append((k, a, b))
            _aggs[f"both__{k}"] = F.sum((F.col(a).isNotNull() & F.col(b).isNotNull()).cast("long"))
            _aggs[f"onlya__{k}"] = F.sum((F.col(a).isNotNull() & F.col(b).isNull()).cast("long"))
            _aggs[f"onlyb__{k}"] = F.sum((F.col(a).isNull() & F.col(b).isNotNull()).cast("long"))
            _aggs[f"lead__{k}"] = F.expr(
                f"percentile_approx(CASE WHEN {a} IS NOT NULL AND {b} IS NOT NULL "
                f"THEN {b} - {a} END, 0.5, 1000)")
    _row = _lab.agg(*[v.alias(k) for k, v in _aggs.items()]).toPandas().iloc[0]

    _pm = pd.DataFrame([{
        "pair": k, "a": a[3:], "b": b[3:],
        "both": int(_row[f"both__{k}"]),
        "only_a": int(_row[f"onlya__{k}"]),
        "only_b": int(_row[f"onlyb__{k}"]),
        "jaccard": _row[f"both__{k}"] / max(1, _row[f"both__{k}"] + _row[f"onlya__{k}"] + _row[f"onlyb__{k}"]),
        "median_months_b_minus_a": _row[f"lead__{k}"],
    } for k, a, b in _pairs])

    _lab = cache_parquet(_lab, "acct_labels")

save(blocks(("counts", _counts), ("pairwise", _pm),
            ("median_mode", kv(("trailing_12_median", _median_mode))),
            ("gates", gates_frame(1))), "m2_closure_definitions")

## M3 · `lead_gap` — how much warning the incumbent rule discards

Brief §9 Step 1, and a deliverable on its own. T0 is the top of the slide: the
most recent month at or before the flag where the rolling-3 average was still
at or above 95% of the trailing-12 median. `lead_gap = flag_month − T0`.

The second block is the part the brief could not compute: the rule against the
**actual** closure flag. "closed, never flagged" is the miss rate on real
attrition; "flagged, never closed" is the false-positive rate that determines
how much of the outreach queue is wasted.

In [ ]:
with timed("M3 lead gap"):
    _flag = _lab.select(F.col(ACCT), F.col("mi_decline30").alias("mi_flag"),
                        F.col("mi_status_c"), F.col("mi_closed_dt")) \
                .where(F.col("mi_flag").isNotNull())

    _top = (_pn.join(_flag.select(ACCT, "mi_flag"), ACCT, "inner")
            .where((F.col("mi") <= F.col("mi_flag")) &
                   (F.col("n_prior") >= CONFIG["BURN_IN_MONTHS"]) &
                   (F.col("avg3") >= CONFIG["T0_MEDIAN_FRAC"] * F.col("med12")))
            .groupBy(ACCT).agg(F.max("mi").alias("mi_t0")))

    _ep = (_flag.join(_top, ACCT, "left")
           .withColumn("lead_gap", F.col("mi_flag") - F.col("mi_t0")))

    _gap_stats = agg1(_ep,
                      episodes=F.count(F.lit(1)),
                      with_t0=F.sum(F.col("mi_t0").isNotNull().cast("long")),
                      p25=F.expr("percentile_approx(lead_gap, 0.25, 1000)"),
                      median=F.expr("percentile_approx(lead_gap, 0.5, 1000)"),
                      p75=F.expr("percentile_approx(lead_gap, 0.75, 1000)"),
                      p90=F.expr("percentile_approx(lead_gap, 0.9, 1000)"),
                      mean=F.avg("lead_gap"))

    _gap_hist = (_ep.where(F.col("lead_gap").isNotNull())
                 .withColumn("gap_bucket",
                             F.when(F.col("lead_gap") <= 0, "0")
                              .when(F.col("lead_gap") <= 2, "1-2")
                              .when(F.col("lead_gap") <= 4, "3-4")
                              .when(F.col("lead_gap") <= 6, "5-6")
                              .when(F.col("lead_gap") <= 9, "7-9").otherwise("10+"))
                 .groupBy("gap_bucket").agg(F.count(F.lit(1)).alias("episodes"))
                 .orderBy("gap_bucket").toPandas())

    # Rule vs. the real label.
    _conf = agg1(_lab,
                 accounts=F.count(F.lit(1)),
                 closed_and_flagged=F.sum((F.col("mi_status_c").isNotNull() &
                                           F.col("mi_decline30").isNotNull()).cast("long")),
                 closed_never_flagged=F.sum((F.col("mi_status_c").isNotNull() &
                                             F.col("mi_decline30").isNull()).cast("long")),
                 flagged_never_closed=F.sum((F.col("mi_status_c").isNull() &
                                             F.col("mi_decline30").isNotNull()).cast("long")),
                 neither=F.sum((F.col("mi_status_c").isNull() &
                                F.col("mi_decline30").isNull()).cast("long")),
                 median_flag_to_close=F.expr(
                     "percentile_approx(CASE WHEN mi_status_c IS NOT NULL AND mi_decline30 IS NOT NULL "
                     "THEN mi_status_c - mi_decline30 END, 0.5, 1000)"))

save(blocks(("lead_gap", _gap_stats), ("lead_gap_hist", _gap_hist),
            ("rule_vs_label", _conf)), "m3_lead_gap")

## P1 · Payments: load, dedup, scope

Order matters. The 5C/6C dedup goes **first** (v3 finding: 6C double-records
inbound RTP, ~21% of 6C rows), then card categories drop, then the semi-join to
the deposit account set. Each filter's effect is reported so a surprise shows
up here rather than as a quiet 20% shortfall in a metric three cells later.

In [ ]:
FLOW_MONTHS = PROFILE_MONTHS          # widen to MONTHS once the profile looks right
_dt_lo, _dt_hi = min(FLOW_MONTHS) + "-01", max(FLOW_MONTHS) + "-31"

with timed("P1 payments load"):
    txn0 = (spark.table(CONFIG["TXN_TABLE"])
            .where((F.col(T["date"]) >= _dt_lo) & (F.col(T["date"]) <= _dt_hi))
            .withColumn("mo", F.substring(F.col(T["date"]), 1, 7))
            .where(F.col("mo").isin(FLOW_MONTHS)))

    _steps = [("0_raw", txn0)]

    txn1 = txn0
    _dedup_note = "skipped"
    if CONFIG["APPLY_DEDUP"]:
        try:
            _dd = spark.table(CONFIG["DEDUP_TABLE"])
            _idcol = T["txn_id"] if T["txn_id"] in _dd.columns else \
                     next(c for c in _dd.columns if "trans_id" in c.lower())
            _flagcol = next((c for c in _dd.columns
                             if re.search(r"(drop|is_dup|dup_flag|exclude)", c, re.I)), None)
            _drop = _dd.where(F.col(_flagcol).cast("boolean")) if _flagcol else _dd
            _drop = _drop.select(F.col(_idcol).alias("_dup_id")).distinct()
            txn1 = txn0.join(_drop, txn0[T["txn_id"]] == F.col("_dup_id"), "left_anti")
            _dedup_note = f"anti-join on {_idcol}" + (f" where {_flagcol}" if _flagcol else " (all rows)")
        except Exception as e:
            _dedup_note = f"UNAVAILABLE: {type(e).__name__}: {str(e)[:120]}"
        _steps.append(("1_dedup_5c6c", txn1))
    gate("P1 dedup applied", not _dedup_note.startswith("UNAVAILABLE"), _dedup_note)

    txn2 = txn1.where(~F.col(T["category"]).isin(CARD_CATS)) if CONFIG["DROP_CARD"] else txn1
    _steps.append(("2_drop_card", txn2))

    txn = txn2.withColumn(
        "cpty_key",
        F.when(F.col(T["category"]).isin(CPTY_HASH_CATS), F.col(T["cpty_hash"]))
         .otherwise(F.coalesce(F.col(T["cpty_acct"]), F.col(T["cpty_hash"]))))
    txn = txn.withColumn("direction",
                         F.when(F.col(T["pays_id"]).isNotNull() & F.col(T["recv_id"]).isNotNull(), "INTERNAL")
                          .when(F.col(T["pays_id"]).isNull() & F.col(T["recv_id"]).isNotNull(), "INBOUND")
                          .when(F.col(T["pays_id"]).isNotNull() & F.col(T["recv_id"]).isNull(), "OUTBOUND")
                          .otherwise("ORPHAN"))

    _rows = [dict(step=n,
                  **{k: v for k, v in zip(["rows", "amount"],
                     s.agg(F.count(F.lit(1)), F.sum(T["amount"])).collect()[0])})
             for n, s in _steps]
    _fil = pd.DataFrame(_rows)
    _fil["rows_kept_pct"] = _fil["rows"] / _fil["rows"].iloc[0] * 100
    _fil["amt_kept_pct"] = _fil["amount"] / _fil["amount"].iloc[0] * 100

    # Null-rule vs category-letter direction: v3 says 1:1. Assertion, not discovery.
    _dir = (txn.withColumn("cat_letter", F.substring(F.col(T["category"]), 2, 1))
            .groupBy("direction", "cat_letter")
            .agg(F.count(F.lit(1)).alias("rows"), F.sum(T["amount"]).alias("amount"))
            .orderBy(F.desc("rows")).limit(30).toPandas())
    _dir["expected"] = _dir["cat_letter"].map(KNOWN["DIRECTION_LETTER"])
    _dir["agrees"] = _dir["expected"] == _dir["direction"]

gate("P1 category letter matches null-rule direction",
     bool(_dir.loc[~_dir["agrees"], "rows"].sum() == 0),
     f"{int(_dir.loc[~_dir['agrees'], 'rows'].sum())} disagreeing rows")

save(blocks(("filters", _fil), ("direction_check", _dir), ("gates", gates_frame(2))),
     "p1_payments_load")

## P2 · Account key compatibility and join yield

Both sides are strings with the same padding, so this is a verification cell,
not a repair one. What it has to establish is the **join yield in dollars**, not
just in rows: if the unmatched legs are disproportionately large-value, the
deposit-scoped view of the graph is biased even at a high row match rate.

In [ ]:
with timed("P2 key compatibility"):
    _dep_keys = acct_dim.select(F.col(ACCT).alias("acct")).distinct()

    def _fmt(sdf, col, label):
        return (sdf.select(F.col(col).cast("string").alias("k")).where(present("k"))
                .agg(F.count(F.lit(1)).alias("n"),
                     F.min(F.length("k")).alias("min_len"), F.max(F.length("k")).alias("max_len"),
                     F.expr("percentile_approx(length(k), 0.5, 100)").alias("median_len"),
                     F.avg(F.col("k").rlike(r"^0").cast("double")).alias("leading_zero_share"),
                     F.avg((~F.col("k").rlike(r"^[0-9]+$")).cast("double")).alias("non_numeric_share"))
                .toPandas().assign(source=label))

    _fmts = pd.concat([_fmt(acct_dim, ACCT, "deposit.acct"),
                       _fmt(txn, T["pays_acct"], "txn.pays_acct"),
                       _fmt(txn, T["recv_acct"], "txn.recv_acct")], ignore_index=True)
    _fmts = _fmts[["source", "n", "min_len", "max_len", "median_len",
                   "leading_zero_share", "non_numeric_share"]]

    def _yield(side):
        col = T["pays_acct"] if side == "pays" else T["recv_acct"]
        s = txn.where(present(col)).select(F.col(col).alias("acct"),
                                           F.col(T["amount"]).alias("amt"))
        j = s.join(_dep_keys.withColumn("m", F.lit(1)), "acct", "left")
        return (j.agg(F.lit(side).alias("side"),
                      F.count(F.lit(1)).alias("legs"),
                      F.avg(F.col("m").isNotNull().cast("double")).alias("leg_match_rate"),
                      (F.sum(F.when(F.col("m").isNotNull(), F.col("amt")).otherwise(0)) /
                       F.sum("amt")).alias("amt_match_rate"),
                      ndv("acct").alias("accounts"))
                .toPandas())

    _yields = pd.concat([_yield("pays"), _yield("recv")], ignore_index=True)

gate("P2 account key formats align",
     bool(_fmts["median_len"].nunique() == 1),
     f"median lengths {_fmts['median_len'].tolist()}")

save(blocks(("formats", _fmts), ("join_yield", _yields), ("gates", gates_frame(1))),
     "p2_key_compatibility")

## P3 · Counterparty identifiability — and a correction to the brief

**`fi_destination_flag` as written in the brief will not work here.**
`cpty_fin_entity_name` is the bank where the *counterparty's account* sits, so
every off-us counterparty carries one and the flag would fire on 100% of
external outflow. The construct the brief actually wants — "money moving to
another bank rather than to a vendor" — needs the **counterparty's own name** to
be a financial institution. Two candidates are measured below:
`fi_by_name` (name matches an FI pattern) and `fi_self` (counterparty name
matches its own bank name). Both are reported so the real base rate is visible
before anything is built on it.

`fin_out_n`, distinct institutions paid, was the best single signal in the
earlier work and comes free from `cpty_fin_entity_name` — that use of the
column is sound, because there it is a *diversity* measure, not a flag.

In [ ]:
with timed("P3 counterparty coverage"):
    _ext = txn.where(F.col("direction").isin("OUTBOUND", "INBOUND"))
    _ext = (_ext.withColumn("cn", norm_name(F.col(T["cpty_name"])))
                .withColumn("own", F.when(F.col("direction") == "OUTBOUND",
                                          norm_name(F.col(T["pays_name"])))
                                    .otherwise(norm_name(F.col(T["recv_name"])))))

    _cov = (_ext.groupBy("direction", T["category"], T["rail"])
            .agg(F.count(F.lit(1)).alias("legs"),
                 F.sum(T["amount"]).alias("amount"),
                 F.avg(present("cpty_key").cast("double")).alias("key_cov"),
                 F.avg(present(T["cpty_name"]).cast("double")).alias("name_cov"),
                 F.avg(present(T["cpty_fi"]).cast("double")).alias("fi_cov"),
                 (F.sum(F.when(present("cpty_key") & present(T["cpty_name"]),
                               F.col(T["amount"])).otherwise(0)) /
                  F.sum(T["amount"])).alias("amt_identifiable"))
            .orderBy(F.desc("amount")).limit(120).toPandas())

    _same = (F.col("cn") == F.col("own")) & present("cn")
    _tax_fp = F.col("cn").rlike(TAX_FP_PATTERN)
    _flags = agg1(_ext.where(F.col("direction") == "OUTBOUND"),
                  out_legs=F.count(F.lit(1)),
                  out_amount=F.sum(T["amount"]),
                  same_name_legs=F.sum(_same.cast("long")),
                  same_name_amt=F.sum(F.when(_same, F.col(T["amount"])).otherwise(0)),
                  same_name_amt_ex_tax=F.sum(F.when(_same & ~_tax_fp, F.col(T["amount"])).otherwise(0)),
                  fi_by_name_amt=F.sum(F.when(F.col("cn").rlike(FI_NAME_PATTERN),
                                              F.col(T["amount"])).otherwise(0)),
                  fi_self_amt=F.sum(F.when(F.col("cn") == norm_name(F.col(T["cpty_fi"])),
                                           F.col(T["amount"])).otherwise(0)),
                  fi_name_present_amt=F.sum(F.when(present(T["cpty_fi"]),
                                                   F.col(T["amount"])).otherwise(0)))

save(blocks(("coverage", _cov), ("outbound_flags", _flags)), "p3_cpty_coverage")

## P4 · Account-month flow panel

Both sides of an internal transaction produce a leg, so a C2C payment appears
once as OUT for the payer and once as IN for the receiver — which is what makes
the panel additive at account level. Off-us share is computed against the
account's own outflow, so it is defined even for accounts with no internal
activity at all.

In [ ]:
with timed("P4 flow panel"):
    _shared = ["mo", "amt", "cat", "rail", "cpty_key", "cpty_name_n", "cpty_fi", "internal"]

    _base = (txn.withColumn("cpty_name_n", norm_name(F.col(T["cpty_name"])))
             .withColumnRenamed(T["amount"], "amt")
             .withColumnRenamed(T["category"], "cat")
             .withColumnRenamed(T["rail"], "rail")
             .withColumnRenamed(T["cpty_fi"], "cpty_fi"))

    _out_leg = (_base.where(present(T["pays_acct"]))
                .withColumn("internal", F.col(T["recv_id"]).isNotNull())
                .withColumn("own_n", norm_name(F.col(T["pays_name"])))
                .select(F.col(T["pays_acct"]).alias("acct"), F.lit("OUT").alias("leg"),
                        "own_n", *_shared))
    _in_leg = (_base.where(present(T["recv_acct"]))
               .withColumn("internal", F.col(T["pays_id"]).isNotNull())
               .withColumn("own_n", norm_name(F.col(T["recv_name"])))
               .select(F.col(T["recv_acct"]).alias("acct"), F.lit("IN").alias("leg"),
                       "own_n", *_shared))
    legs = _out_leg.unionByName(_in_leg)

    if CONFIG["REBUILD"] or not has_parquet("flow_panel"):
        _O = F.col("leg") == "OUT"
        _I = F.col("leg") == "IN"
        _EXT = ~F.col("internal")
        _SAME = (F.col("cpty_name_n") == F.col("own_n")) & present("cpty_name_n") \
                & ~F.col("cpty_name_n").rlike(TAX_FP_PATTERN)

        flow = (legs.groupBy("acct", "mo").agg(
            F.sum(_O.cast("long")).alias("n_out"),
            F.sum(F.when(_O, F.col("amt")).otherwise(0)).alias("amt_out"),
            F.sum((_O & _EXT).cast("long")).alias("n_out_ext"),
            F.sum(F.when(_O & _EXT, F.col("amt")).otherwise(0)).alias("amt_out_ext"),
            F.sum(_I.cast("long")).alias("n_in"),
            F.sum(F.when(_I, F.col("amt")).otherwise(0)).alias("amt_in"),
            F.sum(F.when(_I & _EXT, F.col("amt")).otherwise(0)).alias("amt_in_ext"),
            ndv(F.when(_O & _EXT, F.col("cpty_key"))).alias("cpty_out_n"),
            ndv(F.when(_I & _EXT, F.col("cpty_key"))).alias("cpty_in_n"),
            ndv(F.when(_O & _EXT, F.col("cpty_fi"))).alias("fin_out_n"),
            F.sum(F.when(_O & _SAME, F.col("amt")).otherwise(0)).alias("same_name_out_amt"),
            F.sum((_O & _SAME).cast("long")).alias("same_name_out_n"),
            F.sum(F.when(_O & F.col("cpty_name_n").rlike(FI_NAME_PATTERN),
                         F.col("amt")).otherwise(0)).alias("fi_by_name_out_amt"),
            F.max(F.when(_O, F.col("amt"))).alias("max_single_out"),
        ).withColumn("off_us_out_share",
                     F.when(F.col("amt_out") > 0, F.col("amt_out_ext") / F.col("amt_out")))
         .withColumn("net_flow_ratio",
                     F.when(F.col("amt_in") > 0, (F.col("amt_in") - F.col("amt_out")) / F.col("amt_in"))))
        flow.write.mode("overwrite").partitionBy("mo").parquet(path("flow_panel"))
    flow = spark.read.parquet(path("flow_panel"))

    _f_sum = agg1(flow,
                  account_months=F.count(F.lit(1)),
                  accounts=ndv("acct"),
                  amt_out=F.sum("amt_out"), amt_in=F.sum("amt_in"),
                  median_off_us_share=F.expr("percentile_approx(off_us_out_share, 0.5, 1000)"),
                  months_with_same_name=F.sum((F.col("same_name_out_n") > 0).cast("long")),
                  median_cpty_out_n=F.expr("percentile_approx(cpty_out_n, 0.5, 1000)"),
                  median_fin_out_n=F.expr("percentile_approx(fin_out_n, 0.5, 1000)"))

    _f_mo = (flow.groupBy("mo").agg(F.count(F.lit(1)).alias("account_months"),
                                    F.sum("amt_out").alias("amt_out"),
                                    F.sum("amt_in").alias("amt_in"),
                                    F.avg("off_us_out_share").alias("mean_off_us_share"))
             .orderBy("mo").toPandas())

save(blocks(("flow", _f_sum), ("by_month", _f_mo)), "p4_flow_panel")

## J1 · Deposit ∩ payments coverage

The gate on every payment-derived feature. An account-month with a balance but
no flow row is not "zero flow" — it may be invisible to the graph — and the two
have to be distinguishable before any silence or decline metric is trusted.

In [ ]:
with timed("J1 join coverage"):
    _pj = panel.where(F.col("mo").isin(FLOW_MONTHS)) \
               .select(F.col(ACCT).alias("acct"), "mo", "avg_bal", "eom_bal",
                       "eom_status", "family", "cust")
    joined = _pj.join(flow, ["acct", "mo"], "left")
    joined = cache_parquet(joined, "joined_panel", partition_by="mo")

    _j_sum = agg1(joined,
                  account_months=F.count(F.lit(1)),
                  with_any_flow=F.sum(F.col("n_out").isNotNull().cast("long")),
                  with_outflow=F.sum((F.coalesce("n_out", F.lit(0)) > 0).cast("long")),
                  with_ext_outflow=F.sum((F.coalesce("n_out_ext", F.lit(0)) > 0).cast("long")),
                  accounts=ndv("acct"),
                  accounts_with_flow=ndv(F.when(F.col("n_out").isNotNull(), F.col("acct"))),
                  bal_share_covered=(F.sum(F.when(F.col("n_out").isNotNull(), F.col("avg_bal")).otherwise(0)) /
                                     F.sum("avg_bal")))

    _j_fam = (joined.groupBy("family")
              .agg(F.count(F.lit(1)).alias("account_months"),
                   F.avg(F.col("n_out").isNotNull().cast("double")).alias("flow_coverage"),
                   F.avg("avg_bal").alias("avg_bal"))
              .orderBy(F.desc("account_months")).limit(40).toPandas())

    # Orphans in the other direction: flow rows with no deposit record.
    _orph = agg1(flow.join(_pj.select("acct").distinct().withColumn("m", F.lit(1)),
                           "acct", "left").where(F.col("m").isNull()),
                 orphan_flow_months=F.count(F.lit(1)),
                 orphan_accounts=ndv("acct"),
                 orphan_amt_out=F.sum("amt_out"))

save(blocks(("coverage", _j_sum), ("by_family", _j_fam), ("orphan_flow", _orph)),
     "j1_join_coverage")

## J2 · `cust_pwr_id` ↔ `mdm_id`

Expected one-to-one. Checked through the account bridge rather than asserted,
because a many-to-one on either side changes how account metrics roll up to the
customer level later.

In [ ]:
with timed("J2 customer key cardinality"):
    _acct_mdm = (txn.select(F.col(T["pays_acct"]).alias("acct"), F.col(T["pays_id"]).alias("mdm"))
                 .where(present("acct") & present("mdm"))
                 .union(txn.select(F.col(T["recv_acct"]).alias("acct"),
                                   F.col(T["recv_id"]).alias("mdm"))
                        .where(present("acct") & present("mdm")))
                 .distinct())
    _bridge = (_acct_mdm.join(acct_dim.select(F.col(ACCT).alias("acct"), F.col("cust")),
                              "acct", "inner")
               .select("cust", "mdm").distinct())

    _per_cust = _bridge.groupBy("cust").agg(F.count(F.lit(1)).alias("n_mdm"))
    _per_mdm = _bridge.groupBy("mdm").agg(F.count(F.lit(1)).alias("n_cust"))

    _card = pd.concat([
        agg1(_per_cust, customers=F.count(F.lit(1)),
             cust_with_1_mdm=F.sum((F.col("n_mdm") == 1).cast("long")),
             cust_with_many_mdm=F.sum((F.col("n_mdm") > 1).cast("long")),
             max_mdm_per_cust=F.max("n_mdm")),
        agg1(_per_mdm, mdm_ids=F.count(F.lit(1)),
             mdm_with_1_cust=F.sum((F.col("n_cust") == 1).cast("long")),
             mdm_with_many_cust=F.sum((F.col("n_cust") > 1).cast("long")),
             max_cust_per_mdm=F.max("n_cust"))], ignore_index=True)

    _acct_per_cust = agg1(acct_dim.groupBy("cust").agg(F.count(F.lit(1)).alias("n_acct")),
                          customers=F.count(F.lit(1)),
                          median_accts=F.expr("percentile_approx(n_acct, 0.5, 1000)"),
                          p95_accts=F.expr("percentile_approx(n_acct, 0.95, 1000)"),
                          max_accts=F.max("n_acct"))

gate("J2 cust_pwr_id ↔ mdm_id is 1:1",
     int(_card.loc[_card.metric == "cust_with_many_mdm", "value"].iloc[0]) == 0,
     "many-to-one changes the customer-level roll-up rule")

save(blocks(("cardinality", _card), ("accounts_per_customer", _acct_per_cust),
            ("gates", gates_frame(1))), "j2_customer_keys")

## Z · Answer sheet, gates, timings

One frame to screenshot at the end of a run. `answer` is filled from the cells
above where it can be derived mechanically; the rest is a read-off.

In [ ]:
def _get(name, block, metric, col="value"):
    try:
        f = R[name]
        s = f[(f["block"] == block) & (f["metric"] == metric)][col]
        return s.iloc[0] if len(s) else None
    except Exception:
        return None


_todo = pd.DataFrame([
    dict(q="Q1 Which closure definition should be the label?",
         cell="M2", answer="read pairwise jaccard + median_months_b_minus_a"),
    dict(q="Q2 Do rows continue after closed_dt?",
         cell="D5", answer=_get("d5_closure_behaviour", "rows_after_closed_dt", "rows_after_closed")),
    dict(q="Q2b Accounts still reporting after closed_dt",
         cell="D5", answer=_get("d5_closure_behaviour", "rows_after_closed_dt", "accounts_with_rows_after")),
    dict(q="Q3 What do the non-'C' status codes mean?",
         cell="D3", answer="read by_status; any code with high zero_bal_pct is terminal-like"),
    dict(q="Q4 Are CD closures maturity-driven?",
         cell="D7", answer="read round_tenor_pct by family; >50% means contractual"),
    dict(q="Q5 avg_monthly_bal_1 = MTD or prior month?",
         cell="A1", answer=_get("a1_avg_monthly_bal", "hypotheses", "match_prior_month")),
    dict(q="Q5b  ... constant within month?",
         cell="A1", answer=_get("a1_avg_monthly_bal", "constancy", "constant_within_month")),
    dict(q="Q6 Negative balance prevalence",
         cell="D8", answer="read by_family neg_pct"),
    dict(q="Q7 Counterparty name coverage on outbound",
         cell="P3", answer="read coverage amt_identifiable for OUTBOUND"),
    dict(q="Q8 Rail/category taxonomy stable?",
         cell="P1", answer=_get("p1_payments_load", "gates", None) or "see direction_check"),
    dict(q="Q9 Deposit↔payments join yield (dollars)",
         cell="P2", answer="read join_yield amt_match_rate"),
    dict(q="Q10 lead_gap of the incumbent rule",
         cell="M3", answer=_get("m3_lead_gap", "lead_gap", "median")),
    dict(q="Q11 Closed accounts the rule never flagged",
         cell="M3", answer=_get("m3_lead_gap", "rule_vs_label", "closed_never_flagged")),
    dict(q="Q12 Flow coverage of the deposit book",
         cell="J1", answer=_get("j1_join_coverage", "coverage", "with_any_flow")),
    dict(q="Q13 cust_pwr_id ↔ mdm_id one-to-one?",
         cell="J2", answer=_get("j2_customer_keys", "cardinality", "cust_with_many_mdm")),
])

save(blocks(("answers", _todo),
            ("gates", gates_frame()),
            ("timings", pd.DataFrame(TIMINGS))), "z_answer_sheet")